# FSI: wind load on a tensile membrane (LES ↔ UWM coupling)

Two-way fluid–structure interaction coupling the two repositories:

- **`les_solver.py`** — incompressible LES (Smagorinsky) finite-volume flow solver.
- **`uwm_membrane.py`** — Updated-Weight-Method tensile-membrane form-finding (from `TMS_formfinding`).

**Loop:**
1. Form-find a hypar membrane canopy (UWM) and place it across the channel, facing the wind.
2. Immerse it in the LES flow as a thin **no-slip baffle** (internal CFD faces the membrane crosses become walls).
3. Solve the flow → pressure field; the net pressure force on each blocked face `(p[P]−p[N])·Sf` is the **wind load**.
4. Transfer the load to the membrane nodes and solve the **loaded membrane equilibrium** → the membrane deflects downwind.
5. Re-immerse the deflected shape and repeat (under-relaxed) until the deflection converges.

The channel CFD mesh (`channel_mesh.npz`) is generated automatically on first run.

In [ ]:
import fsi_membrane

# Two-way FSI loop. The channel mesh is auto-generated on first call.
mem, u, p, mesh, g, history = fsi_membrane.run_fsi(
    n_fsi=6,          # FSI outer iterations
    u_wind=1.0,       # inlet wind speed
    nu=0.02,          # kinematic viscosity (Re = u_wind / nu)
    load_scale=1.0,   # wind dynamic-pressure -> membrane load scaling
    relax=0.5,        # under-relaxation of the membrane geometry update
    flow_iters=150,   # LES iterations per FSI cycle
    out="fsi_result.npz",
)

In [ ]:
import numpy as np
defl = np.linalg.norm(mem["coords"] - mem["coords0"], axis=1)
print(f"max membrane deflection = {defl.max():.3f} m")
print(f"net wind force Fx (final) = {history[-1,2]:.3f}")
print(f"deflection update (final FSI iter) = {history[-1,4]:.3e} m  (-> converged)")

In [ ]:
%matplotlib inline
from plot_fsi import plot
plot("fsi_result.npz", "fsi_membrane_wind.png")

from IPython.display import Image
Image("fsi_membrane_wind.png")